KPI 3 - Error Rate


Loading libararies 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest

Loading data sets 

In [3]:
import pandas as pd
data_path = "../../Data/Clean_data/" 

exp    = pd.read_csv(f"{data_path}final_test_data_wrangled.csv")
funnel = pd.read_csv(f"{data_path}web_funnel_sorted.csv")

print("exp shape:", exp.shape)
print("funnel shape:", funnel.shape)

exp shape: (317235, 13)
funnel shape: (744641, 6)


Confrimation Check 

In [4]:
print(funnel.shape)
print(funnel.columns.tolist())
print(funnel["process_step"].unique())  # confirm step names
print(exp["group_test"].unique())       # confirm "Test" / "Control" labels

(744641, 6)
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time', 'source']
['step_3' 'start' 'step_1' 'step_2' 'confirm']
['test' 'control']


Hypothesis:

The New  UI has the same or more errors than the  Old UI 


The New  UI has fewer errors than than Old UI

- H₀: error_rate(Test) ≥ error_rate(Control)   
- H₁: error_rate(Test) < error_rate(Control)   

Error  is defiend as a backward step in the funnel, movingin backwards from step 2 to step 1 indicating user cofusion or friction 

Detecting Errors 

 Sorting  by client and timestamp to preserve step order

 Mapping  steps to numeric order


Detecting Errors 

In [5]:
funnel_sorted = funnel.sort_values(["client_id", "date_time"]).copy()

step_order = {"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4}
funnel_sorted["step_num"] = funnel_sorted["process_step"].map(step_order)

funnel_sorted["prev_step"] = funnel_sorted.groupby("visit_id")["step_num"].shift(1)
funnel_sorted["is_error"]  = funnel_sorted["step_num"] < funnel_sorted["prev_step"]

In [6]:
client_errors = (
    funnel_sorted.groupby("visit_id")["is_error"]
    .any()
    .reset_index()
    .rename(columns={"is_error": "had_error"})
)

client_errors = client_errors.merge(
    exp[["visit_id", "group_test"]].drop_duplicates("visit_id"), 
    on="visit_id", how="inner"
)

In [7]:
print(exp["group_test"].unique())
print(exp["group_test"].value_counts())

['test' 'control']
group_test
test       176699
control    140536
Name: count, dtype: int64


In [8]:
print(client_errors.shape)
print(client_errors["group_test"].value_counts())
print(client_errors["had_error"].value_counts())

(69205, 3)
group_test
test       37085
control    32120
Name: count, dtype: int64
had_error
False    52313
True     16892
Name: count, dtype: int64


 One row per client: did they make at least one error?

In [9]:
client_errors = (
    funnel_sorted.groupby("client_id")["is_error"]
    .any()
    .reset_index()
    .rename(columns={"is_error": "had_error"})
)

Merge with experiment group labels

In [10]:
client_errors = client_errors.merge(
    exp[["client_id", "group_test"]], on="client_id", how="inner"
)
print(client_errors["group_test"].value_counts())
print(client_errors["had_error"].value_counts())

group_test
test       176699
control    140536
Name: count, dtype: int64
had_error
False    170024
True     147211
Name: count, dtype: int64


Group by Error rates 

In [11]:
error_rates = (
    client_errors.groupby("group_test")["had_error"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "errors", "count": "total", "mean": "error_rate"})
)

print(error_rates)



            errors   total  error_rate
group_test                            
control      59547  140536    0.423713
test         87664  176699    0.496121


Testsing 

In [12]:
from statsmodels.stats.proportion import proportions_ztest

test_grp    = client_errors[client_errors["group_test"] == "test"]
control_grp = client_errors[client_errors["group_test"] == "control"]

count = [test_grp["had_error"].sum(), control_grp["had_error"].sum()]
nobs  = [len(test_grp), len(control_grp)]


# alternative='smaller' → H₁: Test error rate < Control error rate
z_stat, p_value = proportions_ztest(count, nobs, alternative="smaller")

print(f"Z-statistic : {z_stat:.4f}")
print(f"P-value     : {p_value:.4f}")
print(f"Alpha       : 0.05")
print()
if p_value < 0.05:
    print("Reject H₀ — Test group has a significantly LOWER error rate.")
else:
    print("Fail to reject H₀ — No significant reduction in error rate.")


Z-statistic : 40.6216
P-value     : 1.0000
Alpha       : 0.05

Fail to reject H₀ — No significant reduction in error rate.


Final Conclusion 

The result supports H₀ — the Test group had a higher error rate than Control, meaning the new UI did not reduce errors and may have actually increased user friction.

Creating  Summary table

In [13]:
error_summary = (
    client_errors.groupby("group_test")["had_error"]
    .agg(
        total_clients   = "count",
        error_count     = "sum",
        no_error_count  = lambda x: (~x).sum(),
        error_rate      = "mean"
    )
    .reset_index()
)

error_summary["error_rate_pct"] = (error_summary["error_rate"] * 100).round(2)

print(error_summary)

  group_test  total_clients  error_count  no_error_count  error_rate  \
0    control         140536        59547           80989    0.423713   
1       test         176699        87664           89035    0.496121   

   error_rate_pct  
0           42.37  
1           49.61  


In [16]:
error_summary.to_csv(f"{data_path}error_summary.csv")